# 리포트 05 — 검출 결과: 어느 조명원으로 어느 거리까지 보이나

> ### ❓ 이 편이 답하는 질문
> **자유공간에서 어느 조명원으로, 수신소자 몇 개로, 드론이 어느 거리까지 보이는가?**

### 결론
1. 선언 예산(EIRP 63 dBm ⟨outputs/report13_freespace.json : meta.link_budget.eirp_dbm⟩) · 베이스라인 500 m ⟨outputs/report13_freespace.json : solve.W1.L_m⟩ · CPI 0.1 s ⟨outputs/report13_freespace.json : solve.W1.T_cpi_s⟩ 에서 공칭 자세 R90(Pd=0.9 가 유지되는 최대 수평거리)은 4.15 km ⟨outputs/report13_freespace.json : ranges.*.R90_C50_m → 5기종×3밴드 최소⟩ ~ 10.04 km ⟨outputs/report13_freespace.json : ranges.*.R90_C50_m → 최대⟩ 다.
2. 그 R90 은 **자세 한 점**의 수다. 헤딩을 균일 평균하면 같은 거리의 Pd 가 0.65 ⟨outputs/report13_freespace.json : ranges.*.E_psi_Pd_at_R90 → 15칸 최대⟩ 이하로 내려간다.
3. 5G SSB 는 PRF 50 Hz ⟨outputs/report13_freespace.json : waveforms.G1.prf_hz⟩ → 프레임 5 ⟨outputs/report13_freespace.json : waveforms.G1.M⟩개라 0-도플러 가드가 도플러 축 전체를 덮는다 — 눈먼 헤딩 비율 1.000 ⟨outputs/report13_freespace.json : ranges.mavic4pro.G1.equal_psd.full_waveform_capture.by_N.1.blind_heading_frac⟩.
4. 파형 우열이 결판나는 축은 σ 를 곱하기 **전**, 기준신호 대역이다(§3.4) — Pd=0.5 에 필요한 출력 SNR 은 WiFi 11.87 dB ⟨outputs/detection_rx_sweep.json : modes.W1.curves.1.snr50⟩ · LTE 13.76 dB ⟨outputs/detection_rx_sweep.json : modes.L1.curves.1.snr50⟩ · 5G 15.03 dB ⟨outputs/detection_rx_sweep.json : modes.G1.curves.1.snr50⟩.
5. 수신소자 N 의 이득 상한은 10log₁₀N 이고 측정 초과분은 최대 0.47 dB ⟨outputs/detection_rx_sweep.json : modes.*.curves.*.snr50 → 9모드×N 최대⟩ — 실이득이 아니라 추정 편향이다.

### ✅ 주장하는 것 / ❌ 주장하지 않는 것

| ✅ 이 편이 주장하는 것 | ❌ 이 편이 주장하지 않는 것 |
|---|---|
| **같은 표적·같은 기하·같은 검출기**에서 잰 세 파형의 상대 비교 — Pfa 를 경험적으로 교정했다 | **절대 검지거리** — σ 레벨이 우리 기하에서 왔다(02 §4). 예산도 선언값이다 |
| 감도사슬의 **항별 분해** — dB 로 닫힌다(합이 출력 SNR 과 일치) | **β>45°** 바이스태틱 — 상반성 잔차가 두 자리 dB 다(§1.1) |
| **β≤45°** 안의 바이스태틱 검지거리 구조와 그 구속 벽(직접파 잔차) | **분산 배치 다중 수신기** — N 은 한 지점 λ/2 ULA 소자 수다(§4) |
| 수신소자 N 의 **이상적 상한**과 측정치가 그 상한에서 벗어난 크기 | 지면 반사·클러터·환경 — 이 편은 자유공간만 푼다 |
| 밴드 간 비교는 **앵커 σ 위에서만**(§3.3) — 우리 기하의 σ(f) 기울기는 쓰지 않는다 | 마이크로도플러·추적 — future work |

### 필요한 사전지식

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| 02 §4 | σ 의 레벨과 주파수 기울기가 왜 측정 앵커에서 오는지 |
| 03 §2 | 조명원 선택의 dB 원장 — 점유 · λ² · 듀티 · PRF |
| 04 | CFAR 문턱, 명목 Pfa 와 경험 Pfa 의 차이, ECA 잔차 |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
# ① σ 격자(자세 × 밴드) — 05 는 읽기만 한다
PYTHONPATH=src ~/.venvs/py312/bin/python src/experiment_freespace_sigma.py
# ② 검지거리 4단계 — 기종마다 1회(결과는 add-only 로 쌓인다)
for D in mini5pro mavic4pro matrice4e phantom4 s1000plus; do \
  PYTHONPATH=src ~/.venvs/py312/bin/python src/experiment_freespace_range.py \
    --stage all --mode W1,L1,G1 --drone $D; done
# ③ 기하·규약 게이트
PYTHONPATH=src:benchmark ~/.venvs/py312/bin/python benchmark/verify_freespace.py
# ④ 파형 9모드 × 수신소자 N 몬테카를로
PYTHONPATH=src ~/.venvs/py312/bin/python src/experiment_detection.py
# ⑤ 그림 8장 + 이 노트북
PYTHONPATH=src ~/.venvs/py312/bin/python src/viz_report05.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/make_report05_results.py
```

| | |
|---|---|
| 출력 | `outputs/report13_freespace.json`, `outputs/report13_sigma_grid.json`, `outputs/detection_rx_sweep.json`, `outputs/verify_freespace.json` |
| 소요 | ② 기종당 565 s ⟨outputs/report13_freespace.json : meta.runtime_s⟩(GPU 2 ⟨outputs/report13_freespace.json : meta.gpus⟩장) · ③ 0.34 s ⟨outputs/verify_freespace.json : meta.runtime_s⟩ · ④ K=6000 ⟨outputs/detection_rx_sweep.json : meta.K⟩ 로 GPU 수십 분 · ⑤ 초 단위 |
| 비고 | ②는 add-only 다 — 한 기종만 다시 돌려도 나머지 칸은 남는다. |

---

## §1. 기하 — 무엇을 어디에 두었나

조명원(TX)과 패시브 수신기(RX)는 지상에 고정, 표적은 두 점의 중점에서 수평거리 `d` 만큼 떨어진 공중에 있다. 아래 값은 전부 **선언**이며 근거문서가 없다. 따라서 이 편의 거리는 *참값*이 아니라 **이 예산 아래의 거리**다.

| 항목 | 값 | 무엇을 정하나 |
|---|---|---|
| 베이스라인 $L$ | 500 m ⟨outputs/report13_freespace.json : solve.W1.L_m⟩ | β(d) 와 직접파 세기 |
| 표적 고도 | 60 m ⟨outputs/report13_freespace.json : solve.W1.alt_m⟩ | 이등분선 앙각 el |
| 장면 방위 $\varphi$ | 90° ⟨outputs/report13_freespace.json : solve.W1.phi_deg⟩ | R1·R2 의 비 |
| EIRP | 63 dBm ⟨outputs/report13_freespace.json : meta.link_budget.eirp_dbm⟩ | DECLARED — no source doc (spec §15-1) ⟨outputs/report13_freespace.json : meta.link_budget.provenance⟩ |
| 수신이득 · NF | 10 dBi ⟨outputs/report13_freespace.json : meta.link_budget.rx_gain_dbi⟩ · 5 dB ⟨outputs/report13_freespace.json : meta.link_budget.noise_figure_db⟩ | 잡음바닥 |
| CPI | 0.1 s ⟨outputs/report13_freespace.json : solve.W1.T_cpi_s⟩ | 프레임 수 M = CPI·PRF |
| 기준채널 | full_waveform_capture ⟨outputs/report13_freespace.json : meta.link_budget.power_normalization.canonical_reference⟩ | 상관에 쓸 수 있는 에너지 — 파일럿만 받는 수신기는 §5 를 볼 것 |

좌표 상수는 `src/freespace_scene.py:72`, 기하 함수는 `src/freespace_scene.py:117`. 기하·규약 게이트 12 ⟨outputs/verify_freespace.json : summary.n_ran⟩건 중 실패 0 ⟨outputs/verify_freespace.json : summary.n_fail⟩건이다.

### §1.1 유효창 — 왜 근거리는 주장에서 뺐나

β 는 근거리에서 커진다. β>45° 인 구간은 `d` < 602 m ⟨outputs/report13_freespace.json : solve.W1.beta_deg → β=45° 보간⟩ 이고, 거기서 SNR 은 이미 58 dB ⟨outputs/report13_freespace.json : solve.W1.snr_d_db → 위 거리에서 보간⟩ 라 검출이 문제되지 않는다. 그 구간을 뺀 이유는 둘 다 **커널 쪽**이다.

| 빼는 이유 | 수치 | 어디서 |
|---|---|---|
| 상반성(정리) 잔차가 크다 | β≤45° 최대 9.34 dB ⟨outputs/sbr_defect_fixes.json : d2_reciprocity_drone.rows → β≤45 행 최대⟩ · 전체 최대 13.69 dB ⟨outputs/sbr_defect_fixes.json : d2_reciprocity_drone.rows → 전 행 최대⟩ | `src/rcs_sbr.py` |
| σ 격자의 고도 행 밖으로 나간다 | el 하한 -20° ⟨outputs/report13_sigma_grid.json : meta.el_deg → 최솟값⟩ · `d` < 126 m ⟨outputs/report13_freespace.json : solve.W1.el_look_deg → el=−20° 보간⟩ 에서 이탈 | `src/experiment_freespace_sigma.py:70` |

헤드라인 거리에서는 β = 2.95° ⟨outputs/report13_freespace.json : solve.W1.beta_deg → R90 에서 보간⟩ 로 **준모노스태틱**이다. σ 는 이등분선 방향의 모노스태틱 값을 쓴다(`src/experiment_freespace_sigma.py:227`).

![geometry](outputs/figures/report05_f1_geometry.png)

**그림 1.** 헤드라인 거리는 바이스태틱 유효창(β≤45°) 안에 있는가?

## §2. 감도사슬 — 출력 SNR 이 어떤 항들의 합인가

d = 1 km · Mavic 4 Pro · 수신소자 1개에서 각 항을 dB 로 적는다. 합이 출력 SNR 이다(점유 규약 equal_psd ⟨outputs/report13_freespace.json : meta.link_budget.power_normalization.canonical_occupancy⟩ — 같은 RE 당 전력).

| 항 | WiFi | LTE | 5G |
|---|---|---|---|
| $\lambda^2$ | -24.80 | -15.77 | -21.34 |
| $\sigma$(공칭 자세) | -9.07 | -25.33 | -13.58 |
| 확산 $1/(4\pi)^3R_1^2R_2^2$ | -153.52 | -153.52 | -153.52 |
| $1/N_0$ | +198.98 | +198.98 | +198.98 |
| CPI | -10.00 | -10.00 | -10.00 |
| **출력 SNR** | +44.59 | +37.35 | +43.53 |

출처 ⟨outputs/report13_freespace.json : ranges.mavic4pro.*.equal_psd.full_waveform_capture.by_N.1.budget_terms_db⟩

⭐ 밴드 간 격차를 만드는 것은 λ² 가 아니라 **σ 항**이다 — 같은 기체·같은 자세에서 세 밴드의 σ 가 16.3 dB ⟨outputs/report13_freespace.json : ranges.mavic4pro.*.budget_terms_db.sigma → 3밴드 최대−최소⟩ 벌어진다. 이것은 파형의 우열이 아니라 **자세 로브 구조**다(§3.3).

![budget](outputs/figures/report05_f2_budget.png)

**그림 2.** 1 km 에서 출력 SNR 은 어떤 항들의 합으로 만들어지는가?

### §2.1 어느 벽이 거리를 정하나

헤드라인 칸의 구속 벽은 열잡음이 아니라 **직접파 잔차**다(dpi_residual ⟨outputs/report13_freespace.json : solve.W1.limit⟩). ECA 억압 깊이를 바꾸면 거리가 이렇게 움직인다.

| ECA 깊이 | R90 |
|---|---|
| 40 dB | 2786 m ⟨outputs/report13_freespace.json : solve.W1.sensitivity_eca_depth.40.R_m⟩ |
| 60 dB | 7618 m ⟨outputs/report13_freespace.json : solve.W1.sensitivity_eca_depth.60.R_m⟩ |
| 90 dB | 9720 m ⟨outputs/report13_freespace.json : solve.W1.sensitivity_eca_depth.90.R_m⟩ |
| 완전 억압 | 9724 m ⟨outputs/report13_freespace.json : solve.W1.sensitivity_eca_depth.inf.R_m⟩ |

예산을 키우면: EIRP 를 70 dBm ⟨outputs/report13_freespace.json : solve.W1.sensitivity_eirp.eirp_dbm[6]⟩ 으로 올리면 14146 m ⟨outputs/report13_freespace.json : solve.W1.sensitivity_eirp.R_thermal_m[6]⟩, CPI 를 0.5 s ⟨outputs/report13_freespace.json : solve.W1.sensitivity_cpi.t_cpi_s[3]⟩ 로 늘리면 14137 m ⟨outputs/report13_freespace.json : solve.W1.sensitivity_cpi.R90_m[3]⟩ 다 — 둘 다 열잡음 축이라 직접파 잔차가 먼저 물리면 소용이 없다.

![walls](outputs/figures/report05_f7_walls.png)

**그림 3.** 베이스라인을 바꾸면 어느 한계가 검지거리를 구속하는가?

## §3. 세 파형 벤치마크

세 조명원은 각 표준이 **늘 켜 두는 기준신호**다 — WiFi VHT-LTF(W1) · LTE CRS(L1) · 5G SSB(G1). 제원은 03 §1, 여기서는 그 셋을 같은 검출기에 물린다.

### §3.1 먼저 문턱을 교정한다

명목 Pfa 를 그대로 쓰면 파형 비교가 성립하지 않는다. CFAR 문턱을 **경험 Pfa 가 목표에 수렴하도록** 잡고, 그 문턱에서 Pd 곡선을 잰다.

| 모드 | 명목 Pfa | 경험 Pfa | 경험/명목 |
|---|---|---|---|
| WiFi | 8.65e-05 ⟨outputs/report13_freespace.json : threshold.pfa.W1.nominal⟩ | 1.01e-04 ⟨outputs/report13_freespace.json : threshold.pfa.W1.empirical⟩ | 1.163 ⟨outputs/report13_freespace.json : threshold.pfa.W1.ratio_emp_over_nominal⟩ |
| LTE | 7.48e-05 ⟨outputs/report13_freespace.json : threshold.pfa.L1.nominal⟩ | 1.02e-04 ⟨outputs/report13_freespace.json : threshold.pfa.L1.empirical⟩ | 1.363 ⟨outputs/report13_freespace.json : threshold.pfa.L1.ratio_emp_over_nominal⟩ |
| 5G | 1.91e-03 ⟨outputs/report13_freespace.json : threshold.pfa.G1.nominal⟩ | 1.04e-04 ⟨outputs/report13_freespace.json : threshold.pfa.G1.empirical⟩ | 0.054 ⟨outputs/report13_freespace.json : threshold.pfa.G1.ratio_emp_over_nominal⟩ |

목표는 1e-04 ⟨outputs/report13_freespace.json : threshold.pfa.W1.target⟩ 다. 5G 의 명목 Pfa 는 경험값의 18 배 ⟨outputs/report13_freespace.json : threshold.pfa.G1.ratio_emp_over_nominal → 역수⟩ 다(04편).

⚠ 5G 의 Pd 곡선 자체는 이 실행에서 재지 못했다 — 그림 4 와 §5 를 볼 것.

![detector](outputs/figures/report05_f3_detector.png)

**그림 4.** 교정된 문턱에서 각 파형이 Pd=0.9 에 필요로 하는 출력 SNR 은 몇 dB 인가?

### §3.2 공칭 자세의 R90, 그리고 그 수가 견디지 못하는 것

5기종 × 3밴드의 R90 은 4.15 km ⟨outputs/report13_freespace.json : ranges.*.R90_C50_m → 최소⟩(mini5pro · WiFi) 에서 10.04 km ⟨outputs/report13_freespace.json : ranges.*.R90_C50_m → 최대⟩(s1000plus · 5G) 사이다. 그러나 이 수는 **자세 한 점**에서 나온다.

| 모드 | 눈먼 헤딩 비율 | 커버리지 상한 | 같은 R90 의 헤딩평균 Pd(Mavic 4 Pro) |
|---|---|---|---|
| WiFi | 0.083 | 0.917 | 0.100 |
| LTE | 0.250 | 0.750 | 0.653 |
| 5G | 1.000 | 0.000 | 0.000 |

출처 ⟨outputs/report13_freespace.json : ranges.mavic4pro.*.equal_psd.full_waveform_capture.by_N.1.blind_heading_frac · coverage_ceiling · E_psi_Pd_at_R90⟩

⭐ 5G 는 **모든 헤딩에서 눈이 먼다**. 거리가 아니라 도플러에서 먼저 죽는다는 뜻이라, 5G 의 R90 은 형식적인 수다. 이 판정의 표적 속도 규약은 5 m/s ⟨outputs/verify_freespace.json : checks.nyquist_fold_check.v_ms⟩ 다.

![range bars](outputs/figures/report05_f4_range_bars.png)

**그림 5.** 공칭 자세에서 기종별·밴드별로 Pd=0.9 가 유지되는 거리는 몇 km 인가?

![heading](outputs/figures/report05_f5_heading.png)

**그림 6.** R90 한 숫자가 표적 헤딩을 균일 평균해도 살아남는가?

### §3.3 ⭐ 밴드 간 비교 — 앵커 σ 위에서만

우리 기하의 σ 주파수 기울기는 측정보다 가파르다(02 §4). 그래서 밴드 비교는 **앵커로 재보정한 σ** 위에서만 말한다. 재보정은 기울기만 측정값으로 돌리고 각도 패턴은 건드리지 않는다 — 정규화 패턴 변화 9.6e-16 dB ⟨outputs/sigma_anchor.json : drones.phantom4.shape_invariance_max_abs_db⟩.

| 기체 | Δσ WiFi | Δσ LTE | Δσ 5G | 보정 후 기울기 | 앵커 비교가능성 |
|---|---|---|---|---|---|
| mini5pro | -2.05 dB | +2.35 dB | -0.30 dB | 0.210 dB/GHz | scaled |
| mavic4pro | -2.27 dB | +1.43 dB | +0.84 dB | 0.210 dB/GHz | scaled |
| matrice4e | -1.68 dB | +0.75 dB | +0.92 dB | 0.210 dB/GHz | scaled |
| phantom4 | -2.32 dB | +2.70 dB | -0.37 dB | 0.210 dB/GHz | direct |
| s1000plus | -2.41 dB | +2.28 dB | +0.13 dB | 0.210 dB/GHz | not_comparable |

출처 ⟨outputs/sigma_anchor.json : drones⟩

이 Δσ 를 R90 으로 옮길 때는 R90 근방의 **국소 지수** 3.992 ⟨outputs/report13_freespace.json : ranges.*.n_local_at_R90 → 15칸 평균⟩ 를 쓴다(`d` 축에서 R ∝ σ^¼ 는 전역적으로 성립하지 않는다, `src/freespace_scene.py:56`).

| 기체 | WiFi | LTE | 5G |
|---|---|---|---|
| mini5pro | 3.69 km | 7.44 km | 5.10 km |
| mavic4pro | 5.81 km | 4.65 km | 6.61 km |
| matrice4e | 4.38 km | 6.34 km | 5.93 km |
| phantom4 | 5.93 km | 5.76 km | 5.65 km |
| s1000plus | 8.46 km | 11.10 km | 10.11 km |

출처 ⟨outputs/sigma_anchor.json : drones.*.modes.slope_only.delta_db⟩

⚠ 결판나지 않는다. 앵커와 **직접 비교 가능한** 유일한 기체(Phantom 4)에서 세 밴드의 폭은 0.83 dB ⟨outputs/sigma_anchor.json : drones.phantom4.modes.slope_only → σ등가 폭⟩ 인데, 앵커가 통제하지 못한 항 하나가 9.50 dB ⟨outputs/sigma_anchor.json : uncontrolled[2].size_db⟩ 다.

밴드 순서는 기체마다 바뀐다. 순서를 만드는 것은 파형이 아니라 **자세별 로브 구조**이고, 앵커는 밴드 평균 레벨만 옮기지 로브를 고치지 않는다.

| 앵커가 통제하지 못한 항 | 상태 | 크기 |
|---|---|---|
| polarisation | UNRESOLVED | 미상 |
| statistic convention (Das mu) | RESOLVED_EMPIRICALLY | +0.93 dB |
| size transfer law | UNRESOLVED | +9.50 dB |
| single platform / single lab | UNRESOLVED | 미상 |
| elevation matching | PARTIAL | +0.73 dB |
| near-field vs far-field, environment | OK | +0.00 dB |

출처 ⟨outputs/sigma_anchor.json : uncontrolled⟩

![anchored bands](outputs/figures/report05_f8_anchored_bands.png)

**그림 7.** 밴드 격차가 앵커의 미통제 항보다 커서 파형의 우열로 읽히는가?

### §3.4 그래서 무엇이 파형 비교로 남나

σ 를 곱하기 **전** 축은 남는다. 같은 표적·같은 기하·같은 검출기에서 Pd=0.5 에 필요한 출력 SNR 이 그것이고, 이 차이는 기준신호 대역과 프레임 수에서 나온다.

| 모드 | 기준신호 | $B_{ref}$ | 프레임 M | Pd=0.5 필요 SNR |
|---|---|---|---|---|
| W1 | VHT-LTF | 76.6 MHz | 112 | 11.87 dB |
| L1 | CRS | 18.0 MHz | 56 | 13.76 dB |
| G1 | SSB | 7.2 MHz | 112 | 15.03 dB |

출처 ⟨outputs/detection_rx_sweep.json : modes⟩

5G 를 세션 신호(NR-PRS, 98.3 MHz ⟨outputs/detection_rx_sweep.json : modes.G3.ref_bw_mhz⟩)까지 열어주면 11.21 dB ⟨outputs/detection_rx_sweep.json : modes.G3.curves.1.snr50⟩ 로 내려간다 — 상시 신호만 쓰는 체제의 대가다(03 §1.1).

⚠ 이 스윕의 CPI·PRF 규약은 §1 과 다르다 — SSB 를 PRF 2000 Hz ⟨outputs/detection_rx_sweep.json : modes.G1.prf⟩ 로 타일링한다(물리값은 50 Hz ⟨outputs/report13_freespace.json : waveforms.G1.prf_hz⟩, §3.2). 그래서 여기서 인용하는 것은 **같은 조건에서의 모드 간 상대 비교**뿐이다.

## §4. 수신소자를 늘리면

N 은 **한 지점의 λ/2 ULA 소자 수**다(`src/experiment_detection.py:181`) — 흩어놓은 N 개의 패시브 수신기가 아니다. 조향벡터는 참 표적 방향에 정확히 맞춰지므로 결과는 **이상적 상한**이다.

| N | 1 | 2 | 3 | 4 |
|---|---|---|---|---|
| 측정 이득 (WiFi) = SNR50(1)−SNR50(N) | +0.00 dB | +3.37 dB | +5.24 dB | +6.44 dB |
| 상한 10log₁₀N | +0.00 dB | +3.01 dB | +4.77 dB | +6.02 dB |

출처 ⟨outputs/detection_rx_sweep.json : modes.W1.curves.*.snr50⟩

9모드 전체에서 상한 초과분은 -0.11 ⟨outputs/detection_rx_sweep.json : modes.*.curves.*.snr50 → 최소⟩ ~ +0.47 dB ⟨outputs/detection_rx_sweep.json : modes.*.curves.*.snr50 → 최대⟩ 다. 결합 잡음전력/σ² = 0.99993 ⟨outputs/detection_rx_sweep.json : modes.W1.combine_ratio⟩ 로 잡음 보존을 확인했다.

![multi rx](outputs/figures/report05_f6_multirx.png)

**그림 8.** 수신소자를 늘렸을 때 얻는 감도는 코히어런트 상한에 얼마나 붙는가?

> ⚠ 이 스윕은 **짧은 베이스라인 벤치 기하**에서 돌았다(`src/experiment_x410.py:101`). 여기서 인용하는 것은 N 사이의 **상대 이득**뿐이고, 절대 SNR50 은 §1 의 자유공간 배치가 아니다.

## §5. 이 편의 한계

| 아직 안 되어 있는 것 | 다음 사람이 이어받을 지점 |
|---|---|
| 앵커 σ 로 자유공간 해를 다시 풀지 않았다 — §3.3 은 국소 지수로 옮긴 1차 전이다 | `src/sigma_anchor.py` 의 보정 σ 를 격자로 써서 `src/experiment_freespace_range.py --stage solve` 재실행 |
| σ 격자 판(2026-07-29T05:36:31 ⟨outputs/report13_sigma_grid.json : meta.generated⟩)이 자유공간 해(2026-07-24T21:45:11 ⟨outputs/report13_freespace.json : meta.generated⟩)보다 나중이다 | 같은 격자로 ②를 다시 돌려 두 시각을 맞춘다 |
| 5G 의 Pd=0.9 문턱은 측정되지 않았다 — outside doppler axis (M=5, \|dopoff\|max=2) ⟨outputs/report13_freespace.json : detector_transfer.S_G.G1.N.1.dopoff.3.reason⟩ | `src/experiment_freespace_range.py` 의 dopoff 격자를 M 인식으로 고치고 재측정. 지금 5G 는 WiFi 에서 잰 문턱을 빌려 쓴다 |
| 파형·수신소자 스윕이 자유공간 배치가 아니고 CPI·PRF 규약도 §1 과 다르다(§3.4 · §4 주의) | `src/experiment_detection.py` 의 X410Scenario 를 `src/freespace_scene.py` 기하로, `CPI_CFG` 를 물리 반복률(`src/freespace_scene.py:233`)로 바꾸면 절대값도 이 편에 들어온다 |
| β>45° 바이스태틱 σ 는 주장 밖이다 | `src/rcs_sbr.py` 의 출사 가시성·대칭화가 정착한 뒤 `benchmark/verify_sbr_defect_fixes.py` 로 잔차 재측정 |
| 기준채널이 full-waveform capture 다 — 파일럿만 받는 수신기는 WiFi 에서 -24.11 dB ⟨outputs/verify_linkbudget.json : BE_processing_gain.waveforms[0].pilot_power_frac_db⟩ 를 잃는다(이 항은 파형 수준 양이라 배치와 무관하다) | `src/experiment_freespace_range.py` 의 CANON_REF 를 pilot_only 로 두고 두 열 병기 |
| 지면 반사·클러터가 빠져 있다(자유공간 FS-1) | `sensitivity.baseline` 의 F⁴ 항을 켜고 FS-3 사다리로 올라간다 |